In [1]:
import pandas as pd
import gseapy as gp
from features_reindex import get_feature, read_data_timecut

In [3]:
dbs = gp.get_library_name() 

In [4]:
for i in dbs:
    if 'reactom' in i.lower():
        print(i)

Reactome_2022
Reactome_Pathways_2024


In [8]:
import requests

def kegg_disease_to_pathways(disease_name: str):
    # 1) find KEGG disease IDs by name
    r = requests.get(f"https://rest.kegg.jp/find/disease/{disease_name}")
    r.raise_for_status()
    ids = [line.split('\t')[0] for line in r.text.strip().splitlines() if line]

    out = {}
    for did in ids[:5]:  # take top matches; adjust as needed
        # 2) link pathways for each disease ID
        q = requests.get(f"https://rest.kegg.jp/link/pathway/{did}")
        if q.status_code == 200 and q.text.strip():
            paths = [row.split('\t')[1].split(':')[-1] for row in q.text.strip().splitlines()]
            out[did] = sorted(set(paths))
    return out

In [9]:
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')
all_df.head(3)

,disease_id,omim,hpo,disease_name,gene_id,score,first_pub_year,last_pub_year,ei,dsi,dpi,uniprot_id,string_id,ori_annotation
0,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,ERBB2,1.0,2004.0,2007.0,0.917,0.298,0.957,P04626,9606.ENSP00000269571,True
1,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,PIK3CA,1.0,2004.0,2023.0,0.978,0.275,0.957,P42336,9606.ENSP00000263967,True
2,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,TP53,1.0,2011.0,2023.0,0.911,0.256,0.957,P04637,9606.ENSP00000269305,True


In [11]:
for disease in all_df['disease_name'].unique():
    print(disease, kegg_disease_to_pathways(disease))

Malignant neoplasm of stomach {}
Malignant neoplasm of colon {}
Malignant melanoma of skin {}
Mesothelioma {}


HTTPError: 400 Client Error: Bad Request for url: https://rest.kegg.jp/find/disease/Cancer,%20Breast

In [2]:
disease = 'ICD10_D83'
feature_list = ['ppi_2019','bioconcept','uniport','esm2']
time = 2019
root = '/itf-fi-ml/shared/users/ziyuzh/svm'
merged_df = None
for feature in feature_list:
    feature_df = get_feature(root, feature)
    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)
    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory

all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')
all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]

df, y = read_data_timecut(disease, all_df, merged_df,time)

In [5]:
import numpy as np

In [7]:
test_idx = df[df['test']==1].index
train_idx = df[y==1].index.difference(test_idx)
df.drop(columns='test', inplace=True)

In [10]:
import pickle
train_pos_df = df.loc[train_idx]
test_pos_df = df.loc[test_idx]
neg_num = 5*len(train_pos_df)

kernel_pkl_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/kerlens/2019.pkl'
with open(kernel_pkl_path, "rb") as f:
    kernels_all_dict = pickle.load(f)
print('loaded kernels: ', list(kernels_all_dict.keys()))

loaded kernels:  ['ppi_2019', 'bioconcept', 'uniport', 'esm2', 'linear_fused', 'geo_fused']


In [11]:
neg_df = df[y == 0]
test_neg_df = neg_df
test_df = pd.concat([test_pos_df, test_neg_df])
test_index_loc = df.index.get_indexer(test_df.index)
y_test = np.array([1] * len(test_pos_df) + [0] * len(test_neg_df))


test_indices = test_df.index.values
enrich_train_genes = train_pos_df.index.values

with open('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2023/name_convert.pkl', 'rb') as file:
    loaded_data = pickle.load(file)
stringId2name,name2stringId,aliases2stringId = loaded_data
del name2stringId,aliases2stringId

input_stringids = enrich_train_genes
gene_names = [stringId2name.get(sid) for sid in input_stringids if stringId2name.get(sid) is not None]
enrich_db = ['GO_Biological_Process_2021','GO_Cellular_Component_2021','GO_Molecular_Function_2021','KEGG_2016']
enr = gp.enrichr(
            gene_list=gene_names,
            gene_sets=enrich_db,
            organism='human', 
            outdir=None
        )
enr_df = enr.results
result_terms = enr_df.loc[enr_df['Adjusted P-value'] < 0.01, ['Gene_set', 'Term']]

In [15]:
stringId2name[test_idx[0]],stringId2name[test_idx[1]]

('SEC61A1', 'DCLRE1C')

In [ ]:
pathway_overlap = []
